In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ===== Path placeholders to update before submitting =====
COMPETITION_DATA_DIR = "/kaggle/input/competitions/llm-classification-finetuning"

# Upload the base model as a Kaggle Dataset, then replace this path.
BASE_MODEL_PATH = "/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/sfairXC__FsfairX-Gemma2-RM-v0.1"

# Upload output/gemma2_qlora_rm/adapter as a Kaggle Dataset, then replace this path.
ADAPTER_PATH = "/kaggle/input/datasets/daidysh643/gemma2-rm-finetune/gemma2_qlora_rm_rep/adapter"

TEST_CSV = f"{COMPETITION_DATA_DIR}/test.csv"
SUBMISSION_CSV = "/kaggle/working/submission.csv"

# ===== Kaggle dual-T4 inference settings: 4bit weights + float16 compute + auto device map =====
MAX_LENGTH = 1800
BATCH_SIZE = 1
DTYPE = "float16"
LOAD_IN_4BIT = True
DEVICE_MAP = "auto"
TTA = True
DISABLE_SOFTCAPPING = True

# If ADAPTER_PATH contains gemma2_qlora_config.json, these are read from it.
CLASSIFIER_HEAD = None  # "mlp" or "linear"
HEAD_DROPOUT = None
HEAD_HIDDEN_RATIO = None

# Optional quick smoke test. Set to None for the real submission.
LIMIT = None

In [ ]:
import inspect
import json
import os
import tempfile
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

In [ ]:
def parse_json_list(value):
    parsed = json.loads(value)
    if not isinstance(parsed, list):
        raise ValueError("Expected a JSON list.")
    return ["" if item is None else str(item) for item in parsed]


def build_compact_pair_text(prompt_value, response_a_value, response_b_value):
    prompts = parse_json_list(prompt_value)
    responses_a = parse_json_list(response_a_value)
    responses_b = parse_json_list(response_b_value)

    turns = []
    for index, prompt in enumerate(prompts):
        response_a = responses_a[index] if index < len(responses_a) else ""
        response_b = responses_b[index] if index < len(responses_b) else ""
        turns.append(
            "<PROMPT>"
            + prompt.strip()
            + "</PROMPT><RESPONSE A>"
            + response_a.strip()
            + "</RESPONSE A><RESPONSE B>"
            + response_b.strip()
            + "</RESPONSE B>"
        )
    return "".join(turns)


def swap_dataframe(df):
    swapped = df.copy()
    swapped["response_a"], swapped["response_b"] = df["response_b"], df["response_a"]
    return swapped


class PreferenceDataset:
    def __init__(self, csv_path, tokenizer, max_length, limit=None, swap_inputs=False):
        df = pd.read_csv(csv_path)
        if limit is not None:
            df = df.head(limit).copy()
        if swap_inputs:
            df = swap_dataframe(df)

        required = ["id", "prompt", "response_a", "response_b"]
        missing = [column for column in required if column not in df.columns]
        if missing:
            raise ValueError(f"{csv_path} is missing columns: {missing}")

        self.examples = [
            {
                "id": str(row["id"]),
                "text": build_compact_pair_text(row["prompt"], row["response_a"], row["response_b"]),
            }
            for _, row in df.iterrows()
        ]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        encoded = self.tokenizer(
            example["text"],
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )
        encoded["id"] = example["id"]
        return encoded


class DataCollatorForPreference:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        ids = [feature.pop("id") for feature in features]
        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        batch["id"] = ids
        return batch

In [ ]:
def torch_dtype(dtype):
    if dtype == "float16":
        return torch.float16
    if dtype == "bfloat16":
        return torch.bfloat16
    if dtype == "float32":
        return torch.float32
    if dtype == "auto":
        return "auto"
    raise ValueError(f"Unsupported dtype: {dtype}")


def maybe_disable_softcapping(config, disable_softcapping):
    if disable_softcapping:
        if hasattr(config, "attn_logit_softcapping"):
            config.attn_logit_softcapping = None
        if hasattr(config, "final_logit_softcapping"):
            config.final_logit_softcapping = None
    return config


class MLPClassificationHead(nn.Module):
    def __init__(self, hidden_size, num_labels, dropout, hidden_ratio):
        super().__init__()
        head_hidden_size = max(num_labels, int(hidden_size * hidden_ratio))
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, head_hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_size, num_labels),
        )

    def forward(self, hidden_states):
        return self.net(hidden_states)


def replace_classification_head(model, head_type, dropout, hidden_ratio):
    if head_type == "linear":
        return model
    if head_type != "mlp":
        raise ValueError(f"Unsupported classifier head: {head_type}")

    old_score_parameter = next(model.score.parameters(), None)
    device = old_score_parameter.device if old_score_parameter is not None else None
    dtype = old_score_parameter.dtype if old_score_parameter is not None else None
    new_score = MLPClassificationHead(model.config.hidden_size, model.config.num_labels, dropout, hidden_ratio)
    if device is not None and dtype is not None:
        new_score = new_score.to(device=device, dtype=dtype)
    model.score = new_score
    return model


@contextmanager
def peft_adapter_with_supported_config(adapter_path, config_cls):
    adapter_dir = Path(adapter_path)
    config_path = adapter_dir / "adapter_config.json"
    if not config_path.exists():
        yield adapter_path
        return

    adapter_config = json.loads(config_path.read_text(encoding="utf-8"))
    supported_keys = set(inspect.signature(config_cls.__init__).parameters)
    supported_keys.discard("self")
    filtered_config = {
        key: value
        for key, value in adapter_config.items()
        if key in supported_keys or key == "peft_type"
    }
    removed_keys = sorted(set(adapter_config) - set(filtered_config))
    if not removed_keys:
        yield adapter_path
        return

    with tempfile.TemporaryDirectory(prefix="peft_adapter_") as tmp:
        tmp_dir = Path(tmp)
        for item in adapter_dir.iterdir():
            target = tmp_dir / item.name
            if item.name == "adapter_config.json":
                continue
            os.symlink(item.resolve(), target, target_is_directory=item.is_dir())
        (tmp_dir / "adapter_config.json").write_text(
            json.dumps(filtered_config, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8",
        )
        print("ignored unsupported PEFT keys:", ", ".join(removed_keys))
        yield str(tmp_dir)

In [ ]:
def load_adapter_config_defaults():
    config_path = Path(ADAPTER_PATH) / "gemma2_qlora_config.json"
    saved_config = {}
    if config_path.exists():
        saved_config = json.loads(config_path.read_text(encoding="utf-8"))

    classifier_head = CLASSIFIER_HEAD or saved_config.get("classifier_head", "mlp")
    head_dropout = HEAD_DROPOUT if HEAD_DROPOUT is not None else float(saved_config.get("head_dropout", 0.1))
    head_hidden_ratio = HEAD_HIDDEN_RATIO if HEAD_HIDDEN_RATIO is not None else float(saved_config.get("head_hidden_ratio", 0.5))
    return classifier_head, head_dropout, head_hidden_ratio


def load_tokenizer(model_path):
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def load_model():
    from peft import LoraConfig, PeftModel
    from transformers import AutoConfig, AutoModelForSequenceClassification, BitsAndBytesConfig

    classifier_head, head_dropout, head_hidden_ratio = load_adapter_config_defaults()
    print("classifier_head", classifier_head)
    print("head_dropout", head_dropout)
    print("head_hidden_ratio", head_hidden_ratio)

    config = AutoConfig.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
    config.num_labels = len(LABEL_COLUMNS)
    config = maybe_disable_softcapping(config, DISABLE_SOFTCAPPING)

    quantization_config = None
    device_map = DEVICE_MAP if LOAD_IN_4BIT else None
    if LOAD_IN_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch_dtype(DTYPE),
            bnb_4bit_use_double_quant=True,
        )
        print("device_map", device_map)

    base = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_PATH,
        config=config,
        trust_remote_code=True,
        torch_dtype=torch_dtype(DTYPE),
        quantization_config=quantization_config,
        device_map=device_map,
        ignore_mismatched_sizes=True,
    )
    if base.config.pad_token_id is None:
        base.config.pad_token_id = base.config.eos_token_id
    base.config = maybe_disable_softcapping(base.config, DISABLE_SOFTCAPPING)
    base = replace_classification_head(base, classifier_head, head_dropout, head_hidden_ratio)

    with peft_adapter_with_supported_config(ADAPTER_PATH, LoraConfig) as adapter_path:
        model = PeftModel.from_pretrained(base, adapter_path)
    if not LOAD_IN_4BIT and torch.cuda.is_available():
        model = model.to("cuda")
    model.eval()
    return model

In [ ]:
def model_input_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def softmax(logits):
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def predict_dataset(model, dataset, tokenizer, batch_size):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=DataCollatorForPreference(tokenizer),
    )
    ids = []
    probabilities = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="predict"):
            row_ids = batch.pop("id")
            device = model_input_device(model)
            batch = {key: value.to(device) for key, value in batch.items()}
            logits = model(**batch).logits.detach().float().cpu().numpy()
            ids.extend(row_ids)
            probabilities.extend(softmax(logits))
    return ids, probabilities

In [ ]:
tokenizer = load_tokenizer(ADAPTER_PATH)
model = load_model()

dataset = PreferenceDataset(TEST_CSV, tokenizer, max_length=MAX_LENGTH, limit=LIMIT)
ids, probabilities = predict_dataset(model, dataset, tokenizer, BATCH_SIZE)

if TTA:
    swapped_dataset = PreferenceDataset(
        TEST_CSV,
        tokenizer,
        max_length=MAX_LENGTH,
        limit=LIMIT,
        swap_inputs=True,
    )
    swapped_ids, swapped_probabilities = predict_dataset(model, swapped_dataset, tokenizer, BATCH_SIZE)
    if ids != swapped_ids:
        raise ValueError("Original and swapped prediction ids do not match.")
    probabilities = [
        (original + swapped[[1, 0, 2]]) / 2.0
        for original, swapped in zip(probabilities, swapped_probabilities)
    ]

submission = pd.DataFrame(
    {
        "id": ids,
        LABEL_COLUMNS[0]: [probs[0] for probs in probabilities],
        LABEL_COLUMNS[1]: [probs[1] for probs in probabilities],
        LABEL_COLUMNS[2]: [probs[2] for probs in probabilities],
    }
)
submission.to_csv(SUBMISSION_CSV, index=False)
print(f"Wrote {SUBMISSION_CSV}")
submission.head()

In [ ]:
submission = pd.read_csv(SUBMISSION_CSV)
expected_columns = ["id", "winner_model_a", "winner_model_b", "winner_tie"]
assert list(submission.columns) == expected_columns, submission.columns.tolist()
assert submission[expected_columns[1:]].notna().all().all()
assert np.isfinite(submission[expected_columns[1:]].to_numpy()).all()
print(submission.shape)
print(submission[expected_columns[1:]].sum(axis=1).describe())
submission.head()